# Ling-3.0-flash on DGX Spark：原生 W4A8 与 Humming + 在线 FP8 LM Head

本 Notebook 演示在 NVIDIA DGX Spark（GB10 / SM121）上从源码部署 Ling-3.0-flash MXFP4，
并对比两套互斥配置：

| | 方案 A（基线） | 方案 B（优化） |
|---|---|---|
| MoE | `flashinfer_mxfp4`（W4A8） | `humming` MXFP4 |
| Dense GEMM | `cutlass` | `cutlass` |
| LM Head | FP32（`--enable-fp32-lm-head`） | 在线量化为动态 FP8 |

两套配置除上述两项外**其余参数完全一致**，可直接做单变量对比。
本 Notebook 不涉及 MTP 投机解码；MTP 的相关改动与实测数据见 `UPSTREAM.md`。

> [!TIP]
> **环境准备：**
> - 推荐 **Python 3.11**、CUDA 13、Ubuntu / DGX OS；
> - 建议用 venv 隔离环境（`python3.11 -m venv venv && source venv/bin/activate`）；
> - 两套服务都用端口 `30000`，**启动另一套前必须先停掉当前服务**。

> [!WARNING]
> 编译缓存与 CUDA、PyTorch、FlashInfer / Humming 版本及 GPU 架构绑定，
> **不要提交到 Git**，应在目标机器上重新生成。

## 步骤 0：配置

后续所有单元都依赖这里定义的变量，**请先执行本单元**。

`MEM_FRACTION_STATIC` 默认取 `0.68`。官方 cookbook 用 `0.75`，但在本机实测中
`0.75` 触发过 GPU OOM 并导致 nvidia 驱动锁死（`rmapiLockAcquire`），只能硬断电恢复。
确认稳定后可自行上调。

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/ly01325/Ling3-Flash-DGX-Spark.git"
BRANCH   = "main"
COMMIT   = ""          # 留空则用分支最新提交；建议发布时固定为完整 SHA

WORKDIR   = Path.home() / "ling3-dgx-spark"
SOURCE_DIR = WORKDIR / "sglang"
VENV_DIR   = WORKDIR / ".venv"
MODEL_DIR  = Path.home() / "models" / "Ling-3.0-flash-fp4"
LOG_DIR    = WORKDIR / "logs"

API_KEY = "sk-ling-cookbook-test"
PORT    = 30000
MEM_FRACTION_STATIC = "0.68"

for key, value in {
    "REPO_URL": REPO_URL, "BRANCH": BRANCH, "COMMIT": COMMIT,
    "WORKDIR": WORKDIR, "SOURCE_DIR": SOURCE_DIR, "VENV_DIR": VENV_DIR,
    "MODEL_DIR": MODEL_DIR, "LOG_DIR": LOG_DIR,
    "API_KEY": API_KEY, "PORT": PORT,
    "MEM_FRACTION_STATIC": MEM_FRACTION_STATIC,
}.items():
    os.environ[key] = str(value)

LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"工作目录: {WORKDIR}")
print(f"源码目录: {SOURCE_DIR}")
print(f"模型目录: {MODEL_DIR}")
print(f"日志目录: {LOG_DIR}")

## 步骤 1：克隆源码

本仓库是 [sgl-project/sglang](https://github.com/sgl-project/sglang) PR
[#33561](https://github.com/sgl-project/sglang/pull/33561)（`ling3-flash-dspark`，
commit `814d2d1`）的快照，另含三处改动：在线 FP8 LM Head、MTP draft 共享 lm_head 模块、
以及一处上游启动崩溃修复。详见 `UPSTREAM.md` 与 `patches/ling3-w4a8-humming.patch`。

在步骤 0 里把 `COMMIT` 填成完整 SHA，即可锁定到确切版本，避免分支后续变动导致无法复现。

In [ ]:
%%bash
set -euo pipefail

mkdir -p "$WORKDIR"
if [[ ! -d "$SOURCE_DIR/.git" ]]; then
  git clone --branch "$BRANCH" "$REPO_URL" "$SOURCE_DIR"
fi

if [[ -n "$COMMIT" ]]; then
  git -C "$SOURCE_DIR" fetch origin "$COMMIT" || git -C "$SOURCE_DIR" fetch origin
  git -C "$SOURCE_DIR" checkout "$COMMIT"
fi

git -C "$SOURCE_DIR" log --oneline -1

## 步骤 2：创建虚拟环境并从源码安装

安装 SGLang 及全量依赖（`[all]`），其中包含 `humming-kernels`。
本步骤**不会**编译依赖模型 shape 的 FlashInfer CUTLASS MXFP4 算子——那在步骤 4。

编译并行度限制为 1，避免 GB10 的统一内存被 Ninja / CMake 并行编译打爆。

In [ ]:
%%bash
set -euo pipefail

python3.11 -m venv "$VENV_DIR"
"$VENV_DIR/bin/python" -m pip install -q --upgrade pip setuptools wheel
MAX_JOBS=1 NINJA_NUM_JOBS=1 CMAKE_BUILD_PARALLEL_LEVEL=1 \
  "$VENV_DIR/bin/pip" install -e "$SOURCE_DIR/python[all]"
"$VENV_DIR/bin/pip" install -q openai modelscope

PYTHONPATH="$SOURCE_DIR/python" "$VENV_DIR/bin/python" - <<'PY'
import importlib.metadata as md
import torch
for pkg in ("sglang", "torch", "flashinfer-python", "humming-kernels"):
    try:
        print(f"{pkg}={md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}=NOT INSTALLED")
print(f"cuda={torch.version.cuda}")
print(f"gpu={torch.cuda.get_device_name(0)}")
print(f"capability={torch.cuda.get_device_capability(0)}")
PY

## 步骤 3：下载 Ling-3.0-flash MXFP4 权重

官方提供预量化的 MXFP4 权重（约 60.5 GB）：

- [ModelScope](https://modelscope.cn/models/inclusionAI/Ling-3.0-flash-fp4)
- [Hugging Face](https://huggingface.co/inclusionAI/Ling-3.0-flash-fp4)

已有权重可跳过本步骤（下方单元会自动检测）。

In [ ]:
%%bash
set -euo pipefail

if [[ -f "$MODEL_DIR/config.json" ]]; then
  echo "复用已有模型: $MODEL_DIR"
else
  "$VENV_DIR/bin/modelscope" download \
    --model inclusionAI/Ling-3.0-flash-fp4 \
    --local-dir "$MODEL_DIR"
fi

## 步骤 4：预编译 FlashInfer CUTLASS MXFP4 算子

**这一步不能跳过。** FlashInfer 需要先探测 GPU 架构和模型 shape 才会生成 `build.ninja`，
所以要分两步：

1. **4.1** 启动一次服务以触发构建规则生成，随后自动停止；
2. **4.2** 找到生成的目录，用 `ninja -j4` 独立完成编译。

> [!WARNING]
> **必须先停掉服务再编译。** 服务加载模型占 66 GB，若与编译同时进行，GB10 的
> 121 GB 统一内存会被打爆——实测触发过两次 `cudafe++ invoked oom-killer` 硬重启。
> 停掉服务后独占内存，`-j4` 峰值约 27 GB，安全且快得多（`-j1` 实测 7 分钟才编完 1/97）。

### 步骤 4.1：触发构建规则生成

下面的单元启动一次服务，最多运行 25 分钟后自动停止。权重加载约需 7 分钟，
随后进入 CUDA Graph 捕获阶段时 FlashInfer 才会生成 `build.ninja`。
看到日志出现 `Capturing batches` 即可，无需等它跑完。

In [ ]:
%%bash
set -euo pipefail

export PYTHONPATH="$SOURCE_DIR/python"
export SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN=1
export SGLANG_JIT_DEEPGEMM_PRECOMPILE=0
export SGLANG_ENABLE_JIT_DEEPGEMM=0
export SGLANG_DSV4_FP4_DEQUANT=0
export SGLANG_FP8_IGNORED_LAYERS=""
export MAX_JOBS=1 NINJA_NUM_JOBS=1 CMAKE_BUILD_PARALLEL_LEVEL=1 FLASHINFER_JIT_MAX_JOBS=1

set +e
timeout --kill-after=30s 25m "$VENV_DIR/bin/python" -m sglang.launch_server \
  --model-path "$MODEL_DIR" --served-model-name ling-v3-flash-fp4 \
  --trust-remote-code --dtype bfloat16 --tp-size 1 --ep-size 1 \
  --host 127.0.0.1 --port 30001 --max-running-requests 1 \
  --max-mamba-cache-size 64 --chunked-prefill-size 8192 \
  --max-prefill-tokens 16384 --page-size 64 --context-length 262144 \
  --cuda-graph-backend-decode full --cuda-graph-max-bs-decode 1 \
  --cuda-graph-bs-decode 1 --cuda-graph-backend-prefill disabled \
  --attention-backend flashinfer --disable-flashinfer-autotune \
  --mem-fraction-static "$MEM_FRACTION_STATIC" --fp8-gemm-backend cutlass \
  --moe-runner-backend flashinfer_mxfp4 \
  --flashinfer-mxfp4-moe-precision default \
  --disable-shared-experts-fusion --enable-fp32-lm-head \
  --json-model-override-args '{"max_position_embeddings":262144,"rope_scaling":{"rope_type":"yarn","factor":2.0,"rope_theta":6000000,"partial_rotary_factor":0.5,"original_max_position_embeddings":131072}}' \
  > "$LOG_DIR/cutlass-cache-generation.log" 2>&1
status=$?
set -e

if [[ $status -ne 0 && $status -ne 124 && $status -ne 137 ]]; then
  echo "触发失败，请检查 $LOG_DIR/cutlass-cache-generation.log"
  tail -20 "$LOG_DIR/cutlass-cache-generation.log"
  exit "$status"
fi
echo "构建规则生成阶段结束，继续执行步骤 4.2。"

### 步骤 4.2：独立编译为 `.so`

缓存目录随 SGLang 版本而变，可能落在下面两处之一，且**与 FlashInfer 版本号绑定**
（升级后旧缓存全部失效，需重编）：

- `~/.cache/flashinfer/<版本>/121a/cached_ops/fused_moe_120/`
- `~/.cache/sglang/.cache/flashinfer/<版本>/121a/cached_ops/fused_moe_120/`

所以这里自动探测而不硬编码路径。编译约 20 分钟。

In [ ]:
import shutil
import subprocess
from pathlib import Path

roots = [
    Path.home() / ".cache" / "flashinfer",
    Path.home() / ".cache" / "sglang" / ".cache" / "flashinfer",
]
candidates = [p for r in roots for p in r.glob("*/121a/cached_ops/fused_moe_120/build.ninja")]
if not candidates:
    raise RuntimeError(
        "未找到 fused_moe_120/build.ninja。请先执行步骤 4.1，"
        f"并检查 {LOG_DIR/'cutlass-cache-generation.log'}"
    )

build_dir = max(candidates, key=lambda p: p.stat().st_mtime).parent
print(f"编译目录: {build_dir}")

running = subprocess.run(["pgrep", "-f", "sglang.launch_server"], capture_output=True)
if running.returncode == 0:
    raise RuntimeError("检测到 sglang 服务仍在运行。请先停止，否则编译可能触发 OOM。")

subprocess.run([shutil.which("ninja") or "ninja", "-j4", "-C", str(build_dir)], check=True)
for so in build_dir.glob("*.so"):
    print(f"{so.name}: {so.stat().st_size / 1048576:.1f} MB")

## 步骤 5：准备 Humming kernel

`humming-kernels` 已随步骤 2 一并安装。这里预先构建它的通用 launcher 与 NVRTC helper；
依赖模型 shape 的 MoE CUBIN 会在首次启动方案 B 时由 NVRTC 自动编译（单个 kernel 秒级），
并缓存到 `~/.humming/cache`，**不需要像 FlashInfer 那样单独预编译**。

另外 GB10 使用统一 LPDDR5X 内存，NVML 读不到独立显存时钟，而 Humming 依赖内存带宽
做 roofline 选型，因此需要 **273 GB/s** 的 GB10 fallback。下面的单元会检查并按需补上。

In [ ]:
from pathlib import Path

import humming.utils.device as humming_device
from humming.ops.utils import init_humming_launcher
from humming.utils.nvrtc import may_build_nvrtc_compile_binary

init_humming_launcher()
may_build_nvrtc_compile_binary()
print("Humming launcher 与 NVRTC helper 已准备完成")

device_file = Path(humming_device.__file__)
source = device_file.read_text()

if '"GB10" in gpu_name' in source and "return 273.0" in source:
    print(f"GB10 273 GB/s fallback 已存在: {device_file}")
else:
    old = """        mem_clock_mhz = pynvml.nvmlDeviceGetMaxClockInfo(
            handle, pynvml.NVML_CLOCK_MEM
        )"""
    new = """        try:
            mem_clock_mhz = pynvml.nvmlDeviceGetMaxClockInfo(
                handle, pynvml.NVML_CLOCK_MEM
            )
        except pynvml.NVMLError_NotSupported:
            # GB10 uses unified LPDDR5X memory and does not expose a memory
            # clock through NVML. Its documented peak bandwidth is 273 GB/s.
            if "GB10" in gpu_name:
                return 273.0
            raise"""
    if old not in source:
        raise RuntimeError(f"无法自动补丁，请手工检查 {device_file}")
    device_file.write_text(source.replace(old, new))
    print(f"已写入 GB10 273 GB/s fallback: {device_file}")

## 步骤 6：启动方案 A —— 原生 W4A8（基线）

- MoE：`flashinfer_mxfp4`（W4A8）
- Dense GEMM：`cutlass`
- LM Head：FP32（`--enable-fp32-lm-head`）
- 不启用 Humming、在线 FP8 LM Head 或 MTP

服务在**后台**运行，PID 写入 `$WORKDIR/sglang-server.pid`，日志在 `$LOG_DIR` 下。
启动约需 8 分钟（权重加载 ~7 分钟 + CUDA Graph 捕获）。

In [ ]:
%%bash
set -euo pipefail

export PYTHONPATH="$SOURCE_DIR/python"
export SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN=1
export SGLANG_ENABLE_FP8_LM_HEAD=0
export SGLANG_JIT_DEEPGEMM_PRECOMPILE=0
export SGLANG_ENABLE_JIT_DEEPGEMM=0
export SGLANG_DSV4_FP4_DEQUANT=0
export SGLANG_FP8_IGNORED_LAYERS=""
export MAX_JOBS=1 NINJA_NUM_JOBS=1

setsid nohup "$VENV_DIR/bin/python" -m sglang.launch_server \
  --model-path "$MODEL_DIR" --served-model-name ling-v3-flash-fp4 \
  --trust-remote-code --dtype bfloat16 --tp-size 1 --ep-size 1 \
  --host 0.0.0.0 --port "$PORT" --api-key "$API_KEY" \
  --max-running-requests 1 --max-mamba-cache-size 64 \
  --chunked-prefill-size 8192 --max-prefill-tokens 16384 \
  --page-size 64 --context-length 262144 \
  --cuda-graph-backend-decode full --cuda-graph-max-bs-decode 1 \
  --cuda-graph-bs-decode 1 --cuda-graph-backend-prefill disabled \
  --random-seed 308534008 --reasoning-parser deepseek-r1 \
  --tool-call-parser qwen25 --attention-backend flashinfer \
  --disable-flashinfer-autotune \
  --mem-fraction-static "$MEM_FRACTION_STATIC" --fp8-gemm-backend cutlass \
  --moe-runner-backend flashinfer_mxfp4 \
  --flashinfer-mxfp4-moe-precision default \
  --disable-shared-experts-fusion --enable-fp32-lm-head \
  --json-model-override-args '{"max_position_embeddings":262144,"rope_scaling":{"rope_type":"yarn","factor":2.0,"rope_theta":6000000,"partial_rotary_factor":0.5,"original_max_position_embeddings":131072}}' \
  > "$LOG_DIR/native-w4a8-server.log" 2>&1 < /dev/null &

echo $! > "$WORKDIR/sglang-server.pid"
echo "方案 A 已在后台启动，PID=$(<"$WORKDIR/sglang-server.pid")"
echo "日志: $LOG_DIR/native-w4a8-server.log"

### 停止当前服务（切换配置前必须执行）

两套配置都用端口 `30000`，启动另一套前先跑这个单元。

In [ ]:
%%bash
set -euo pipefail

pid_file="$WORKDIR/sglang-server.pid"
if [[ -f "$pid_file" ]]; then
  pid=$(<"$pid_file")
  if kill -0 "$pid" 2>/dev/null; then
    kill -- -"$pid" 2>/dev/null || kill "$pid" 2>/dev/null || true
    for _ in $(seq 1 60); do
      kill -0 "$pid" 2>/dev/null || break
      sleep 1
    done
  fi
  rm -f "$pid_file"
fi

pkill -f "[s]glang.launch_server" 2>/dev/null || true
sleep 3
echo "服务已停止。当前内存:"
free -h | head -2 | tail -1

## 步骤 7：启动方案 B —— Humming MoE + 在线 FP8 LM Head

**启动前请先执行上面的「停止当前服务」单元。**

- MoE：`humming` MXFP4
- Dense GEMM：`cutlass`
- LM Head：`SGLANG_ENABLE_FP8_LM_HEAD=1`，加载 BF16 权重后在线量化为动态 FP8
- 其余参数与方案 A **完全一致**，便于做严格的单变量对比

首次启动时 Humming 会用 NVRTC 编译若干 MoE kernel（秒级，缓存到 `~/.humming/cache`），
之后的启动会直接命中缓存。

In [ ]:
%%bash
set -euo pipefail

export PYTHONPATH="$SOURCE_DIR/python"
export SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN=1
export SGLANG_ENABLE_FP8_LM_HEAD=1
export SGLANG_JIT_DEEPGEMM_PRECOMPILE=0
export SGLANG_ENABLE_JIT_DEEPGEMM=0
export SGLANG_DSV4_FP4_DEQUANT=0
export SGLANG_FP8_IGNORED_LAYERS=""
export HUMMING_COMPILER=nvrtc
export HUMMING_CACHE_DIR="$HOME/.humming/cache"
export MAX_JOBS=1 NINJA_NUM_JOBS=1

setsid nohup "$VENV_DIR/bin/python" -m sglang.launch_server \
  --model-path "$MODEL_DIR" --served-model-name ling-v3-flash-fp4 \
  --trust-remote-code --dtype bfloat16 --tp-size 1 --ep-size 1 \
  --host 0.0.0.0 --port "$PORT" --api-key "$API_KEY" \
  --max-running-requests 1 --max-mamba-cache-size 64 \
  --chunked-prefill-size 8192 --max-prefill-tokens 16384 \
  --page-size 64 --context-length 262144 \
  --cuda-graph-backend-decode full --cuda-graph-max-bs-decode 1 \
  --cuda-graph-bs-decode 1 --cuda-graph-backend-prefill disabled \
  --random-seed 308534008 --reasoning-parser deepseek-r1 \
  --tool-call-parser qwen25 --attention-backend flashinfer \
  --disable-flashinfer-autotune \
  --mem-fraction-static "$MEM_FRACTION_STATIC" --fp8-gemm-backend cutlass \
  --moe-runner-backend humming --flashinfer-mxfp4-moe-precision default \
  --disable-shared-experts-fusion \
  --json-model-override-args '{"max_position_embeddings":262144,"rope_scaling":{"rope_type":"yarn","factor":2.0,"rope_theta":6000000,"partial_rotary_factor":0.5,"original_max_position_embeddings":131072}}' \
  > "$LOG_DIR/humming-fp8-lm-head-server.log" 2>&1 < /dev/null &

echo $! > "$WORKDIR/sglang-server.pid"
echo "方案 B 已在后台启动，PID=$(<"$WORKDIR/sglang-server.pid")"
echo "日志: $LOG_DIR/humming-fp8-lm-head-server.log"

## 步骤 8：确认优化确实生效

**静默降级是这类部署最常见的坑**：环境变量拼错、或分支缺少补丁时，SGLang 不会报错，
只是安静地跑在未优化的路径上——性能差一截却毫无提示。

下面的单元等待服务就绪，然后逐项核对方案 B 的三个标志。

In [ ]:
import re
import time
from pathlib import Path

log_path = LOG_DIR / "humming-fp8-lm-head-server.log"

for _ in range(240):
    if log_path.exists() and "ready to roll" in log_path.read_text(errors="replace"):
        break
    time.sleep(10)
else:
    raise RuntimeError(f"服务在 40 分钟内未就绪，请检查 {log_path}")

log = log_path.read_text(errors="replace")

checks = {
    "Humming MoE backend": bool(re.search(r"moe_runner_backend=humming", log)),
    "Humming quant method": "Mxfp4HummingMoEMethod" in log,
    "在线 FP8 LM Head": "Online FP8 quantization enabled for lm_head" in log,
}
for name, ok in checks.items():
    print(f"{'[OK]  ' if ok else '[FAIL]'} {name}")

if not all(checks.values()):
    raise RuntimeError(
        "有优化项未生效。若 FP8 LM Head 失败，通常是所用分支缺少对应补丁 "
        "（SGLANG_ENABLE_FP8_LM_HEAD 不是上游变量，设置未知变量不会报错）。"
    )
print("\n三项检查全部通过。")

## 步骤 9：健康检查与流式推理

下面执行一次流式请求，报告 TTFT、端到端耗时和近似 decode 吞吐。

> 这只是一次冒烟测试。**正式性能对比请使用固定输入/输出长度的 benchmark**，
> 并注意：decode 吞吐对采样参数敏感——greedy 与
> `temperature=0.6 / top_p=0.95 / top_k=29` 的结果不可直接互比（详见 `UPSTREAM.md`）。

In [ ]:
import time
import urllib.request

from openai import OpenAI

models_url = f"http://127.0.0.1:{PORT}/v1/models"
headers = {"Authorization": f"Bearer {API_KEY}"}
for attempt in range(180):
    try:
        req = urllib.request.Request(models_url, headers=headers)
        with urllib.request.urlopen(req, timeout=5) as resp:
            print(resp.read().decode("utf-8"))
        break
    except Exception:
        if attempt == 179:
            raise RuntimeError("服务在 30 分钟内未就绪，请检查 server log。")
        time.sleep(10)

client = OpenAI(base_url=f"http://127.0.0.1:{PORT}/v1", api_key=API_KEY)

start = time.perf_counter()
first_token_at = None
completion_tokens = None
parts = []

stream = client.chat.completions.create(
    model="ling-v3-flash-fp4",
    messages=[{"role": "user", "content": "请计算 17 × 23，并给出简洁推导。"}],
    temperature=0.6,
    top_p=0.95,
    max_tokens=2048,
    stream=True,
    stream_options={"include_usage": True},
    extra_body={"top_k": 29, "chat_template_kwargs": {"enable_thinking": True}},
)

for chunk in stream:
    if chunk.usage:
        completion_tokens = chunk.usage.completion_tokens
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    piece = (getattr(delta, "reasoning_content", None) or delta.content or "")
    if piece:
        first_token_at = first_token_at or time.perf_counter()
        parts.append(piece)

end = time.perf_counter()
ttft = (first_token_at - start) if first_token_at else float("nan")
decode_s = (end - first_token_at) if first_token_at else float("nan")
tps = completion_tokens / decode_s if completion_tokens and decode_s > 0 else float("nan")

print(f"TTFT: {ttft * 1000:.2f} ms")
print(f"End-to-end latency: {end - start:.2f} s")
print(f"Completion tokens: {completion_tokens}")
print(f"Approx. decode throughput: {tps:.2f} token/s")
print("".join(parts))

## 步骤 10：复现记录清单

对外发布结果前请补齐：

- 源码仓库 URL、分支与**完整 commit SHA**；
- `pip freeze`、CUDA Toolkit、NVIDIA driver 版本与 GPU 型号；
- 模型仓库 revision 或权重校验值；
- FlashInfer CUTLASS `.so` 与 Humming CUBIN 均在目标机器上成功生成；
- 两套配置使用**相同的** prompt、采样参数、并发度与随机种子；
- 对 FP8 LM Head 单独做真实数据集的准确率对比（本仓库的 GSM8K-1319 结果见 `UPSTREAM.md`）。

**可以提交到 Git**：Notebook、源码补丁、依赖锁文件。
**不要提交**：模型权重、`.venv`、FlashInfer 缓存、Humming CUBIN——它们与
CUDA / PyTorch / FlashInfer 版本及 GPU 架构绑定，必须在目标机器上重新生成。